# PANDAS II — COMPLEMENTO CODEX

#### Nota Codex — Correspondencia con el curso

Acompaña [Pandas II: indexación](04_Pandas_Base_II_Indexacion.ipynb). Las secciones 1 a 3 refuerzan loc, iloc y filtros; la 4 muestra cómo escribir una asignación sin ambigüedad; la 5 vuelve al MultiIndex ya presentado en el curso. Reutiliza la idea de alineación del complemento I, sin exigir que su kernel siga abierto.

## Objetivos de aprendizaje

1. Elegir etiquetas o posiciones de manera consciente.
2. Anticipar extremos incluidos, estructuras devueltas y errores habituales.
3. Construir máscaras legibles y reconocer su alineación.
4. Diferenciar un alias de una copia y una selección de una asignación.

## Preparación

In [1]:
import pandas as pd
from IPython.display import display
ventas = pd.DataFrame({
    'ciudad': ['Lima', 'Quito', 'Lima', 'Bogotá'],
    'unidades': [2, 5, 1, 4],
    'precio': [10.0, 8.0, 12.0, 9.0]
}, index=[10, 20, 40, 80])
display(ventas)

,ciudad,unidades,precio
10,Lima,2,10.0
20,Quito,5,8.0
40,Lima,1,12.0
80,Bogotá,4,9.0


## 1. Etiqueta 20 no significa posición 20

#### Nota Codex — Concepto y uso

`loc[fila, columna]` interpreta etiquetas; `iloc[fila, columna]` interpreta posiciones desde cero. Los números del índice pueden saltar y seguir siendo etiquetas perfectamente válidas.

| Pregunta | Expresión | Resultado esperado |
|---|---|---|
| Fila etiquetada 20 | ventas.loc[20] | Series de Quito |
| Segunda fila | ventas.iloc[1] | Series de Quito |
| Una tabla con esa fila | ventas.loc[[20]] | DataFrame de una fila |
| Precio de esa fila | ventas.loc[20, 'precio'] | Escalar 8.0 |

In [2]:
display(ventas.loc[20])
display(ventas.iloc[1])
display(ventas.loc[[20]])
print(ventas.loc[20, 'precio'])
# at e iat son atajos de acceso a una celda en esta tabla con etiquetas únicas.
print(ventas.at[20, 'precio'], ventas.iat[1, 2])

8.0
8.0 8.0


ciudad      Quito
unidades        5
precio        8.0
Name: 20, dtype: object

ciudad      Quito
unidades        5
precio        8.0
Name: 20, dtype: object

,ciudad,unidades,precio
20,Quito,5,8.0


#### Nota Codex — Lectura del resultado

loc[20] e iloc[1] coinciden en esta tabla, pero loc[1] buscaría una etiqueta inexistente. Con etiquetas únicas, at accede por etiqueta e iat por posición a un escalar; loc e iloc siguen siendo suficientes para resolver el curso.

## 2. Rangos, orden y errores controlados

#### Nota Codex — Concepto y uso

`loc[10:40]` incluye las etiquetas extremas; `iloc[0:2]` excluye la posición 2. Para entender un corte por etiqueta, mira el orden del índice. Este ejemplo usa un índice ordenado y único. Los índices desordenados o duplicados pueden complicar los cortes; una lista explícita de etiquetas evita ambigüedades cuando sabes cuáles quieres.

Una selección escalar fuera de rango es un error; un slice posicional se recorta hasta donde existan filas.

In [3]:
print('Etiquetas con loc:', ventas.loc[10:40].index.tolist())
print('Etiquetas con iloc:', ventas.iloc[0:2].index.tolist())
print('Última fila:', ventas.iloc[-1].name)
display(ventas.loc[[80, 10], ['precio', 'ciudad']])
print('Slice hasta 99:', len(ventas.iloc[:99]))
try:
    ventas.loc[99]
except KeyError:
    print('KeyError: no existe la etiqueta 99.')
try:
    ventas.iloc[99]
except IndexError:
    print('IndexError: no existe la posición 99.')

Etiquetas con loc: [10, 20, 40]
Etiquetas con iloc: [10, 20]
Última fila: 80
Slice hasta 99: 4
KeyError: no existe la etiqueta 99.
IndexError: no existe la posición 99.


,precio,ciudad
80,9.0,Bogotá
10,10.0,Lima


#### Nota Codex — Lectura del resultado

loc devuelve [10, 20, 40]; iloc devuelve [10, 20]. La selección explícita [80, 10] también cambia el orden de salida. Los errores intencionales se capturan para compararlos; un filtro vacío, en cambio, es un resultado válido.

## 3. Máscaras legibles y alineadas

#### Nota Codex — Concepto y uso

Una máscara responde sí o no para cada fila. `isin(lista)` comprueba pertenencia y `between(a, b)` incluye ambos límites de manera predeterminada. `~` niega una condición. Son formas de seleccionar, no de limpiar datos.

Construye las máscaras a partir de la tabla que vas a filtrar. Una Series booleana usada con loc se alinea por etiqueta; un array booleano se aplica por posición. No reutilices una máscara de otra tabla sin comprobar esa correspondencia.

In [4]:
ciudades = ventas['ciudad'].isin(['Lima', 'Quito'])
cantidades = ventas['unidades'].between(2, 5)
mascara = ciudades & cantidades
display(mascara)
display(ventas.loc[mascara, ['ciudad', 'unidades']])
display(ventas.loc[~ciudades])
sin_coincidencias = ventas.loc[ventas['unidades'] > 100]
print('Resultado vacío:', sin_coincidencias.empty)
# Cambiar el orden de la máscara conserva sus asociaciones etiqueta-booleano.
invertida = mascara.iloc[::-1]
display(ventas.loc[invertida])
assert ventas.loc[invertida].index.tolist() == [10, 20]

Resultado vacío: True


10     True
20     True
40    False
80    False
dtype: bool

,ciudad,unidades
10,Lima,2
20,Quito,5


,ciudad,unidades,precio
80,Bogotá,4,9.0


,ciudad,unidades,precio
10,Lima,2,10.0
20,Quito,5,8.0


#### Nota Codex — Lectura del resultado

Quedan las etiquetas 10 y 20: Lima con 2 unidades y Quito con 5. Al invertir la Series booleana se conservan esas etiquetas. Convertir esa máscara invertida a NumPy cambiaría las asociaciones porque perdería el índice. Los paréntesis son necesarios si escribes las comparaciones directamente junto a & o |.

## 4. Seleccionar, copiar y asignar

#### Nota Codex — Concepto y uso

`otra = ventas` crea un segundo nombre para el mismo objeto; no crea una tabla independiente. Para preparar una variante de trabajo usa `.copy()`. Para modificar el destino elegido escribe la selección y la asignación en **una sola llamada loc**.

Evita `df[condicion]['columna'] = valor`: esa asignación encadenada no expresa de manera fiable que quieres cambiar df y no funciona como actualización del original con Copy-on-Write. Esta regla es útil tanto en Pandas 2 como en Pandas 3. Aquí solo se practica la mecánica; las decisiones de corrección de datos pertenecen a Pandas V.

In [5]:
alias = ventas
copia = ventas.copy()
print('Alias es el mismo objeto:', alias is ventas)
print('Copia es el mismo objeto:', copia is ventas)
copia.loc[copia['ciudad'] == 'Lima', 'unidades'] = 9
display(copia)
print('Original sin modificar:', ventas['unidades'].tolist())
subtabla = ventas.loc[ventas['ciudad'] == 'Lima', ['unidades']].copy()
subtabla.loc[10, 'unidades'] = 7
assert ventas.loc[10, 'unidades'] == 2

Alias es el mismo objeto: True
Copia es el mismo objeto: False
Original sin modificar: [2, 5, 1, 4]


,ciudad,unidades,precio
10,Lima,9,10.0
20,Quito,5,8.0
40,Lima,9,12.0
80,Bogotá,4,9.0


#### Nota Codex — Lectura del resultado

Solo la copia tiene 9 unidades en Lima. La subtabla también puede modificarse de forma explícita sin alterar ventas. No necesitas conocer las optimizaciones internas de memoria para expresar qué objeto quieres modificar.

## 5. Etiquetas repetidas y MultiIndex

#### Nota Codex — Concepto y uso

Un índice puede repetir etiquetas. Por eso `loc['Lima']` devuelve aquí varias filas. La unicidad es una propiedad de tus datos, no una garantía automática de set_index.

Un MultiIndex tiene niveles. Una selección parcial elimina el nivel ya especificado; una selección completa usa una tupla. `reset_index()` devuelve los niveles a columnas. No agrega, no cruza tablas ni elimina registros.

In [6]:
por_ciudad = ventas.set_index('ciudad')
print('¿Índice único?', por_ciudad.index.is_unique)
display(por_ciudad.loc['Lima'])
detalle = pd.DataFrame({
    'cliente': ['A', 'A', 'B'],
    'producto': ['libro', 'lápiz', 'libro'],
    'unidades': [2, 3, 1]
}).set_index(['cliente', 'producto'])
display(detalle.loc['A'])
print('Una compra:', detalle.loc[('A', 'libro'), 'unidades'])
display(detalle.xs('libro', level='producto'))
display(detalle.reset_index())

¿Índice único? False
Una compra: 2


,unidades,precio
ciudad,,
Lima,2,10.0
Lima,1,12.0


,unidades
producto,
libro,2
lápiz,3


,unidades
cliente,
A,2
B,1


,cliente,producto,unidades
0,A,libro,2
1,A,lápiz,3
2,B,libro,1


#### Nota Codex — Lectura del resultado

Lima tiene dos filas: una etiqueta no implica necesariamente un único registro. La pareja (A, libro) sí es única en detalle y devuelve 2 unidades. xs muestra los dos clientes que compraron libro, sin inventar compras ausentes.

#### Nota Codex — Repaso y siguiente paso

Antes de ejecutar una selección, anticipa etiquetas, cantidad de filas y tipo del resultado. En Pandas III esa misma atención te ayudará a comprobar si importaste la estructura esperada.

#### Nota Codex — Límites del complemento

El diagnóstico de nulos, duplicados, frecuencias y estadísticos se estudia en **Pandas IV**.
La conversión y limpieza de columnas, imputación, recodificación y tratamiento de texto se estudian en **Pandas V**.
`concat` y `merge` se estudian en **Pandas VI**.
Los gráficos y su preparación están en [Visualización con Pandas](../../18_VISUALIZACION/02_PANDAS/01_Notebooks/04_Visualization_II_Pandas.ipynb). Aquí no se repiten esos temas.